# E3 BioBERT Action Classifier

CLEF SimpleText Task 1.1 sentence-level biomedical text simplification.

This notebook trains a biomedical sentence classifier on the provided Cochrane-auto action labels, then uses the predicted action to decide what should happen to each sentence in a later simplification pipeline.

## Objective

Train a sentence-level no-context classifier for four simplification actions:

- `rephrase` -> send sentence to BioBART simplification
- `delete` -> remove sentence from output
- `ignore` -> keep original sentence unchanged
- `split` -> route to split handling; for now send to BioBART, future dedicated split model

The classifier uses only the `complex` sentence as input. It does not use neighbouring sentences, document position, paragraph context, or the `simple` reference.

Evaluation is classification-based: Accuracy, Precision Macro, Recall Macro, F1 Macro, and F1 Weighted. SARI/BLEU/BERTScore apply only after a full text generation pipeline produces simplified text.

## Setup

In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path
from typing import Any

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

sns.set_theme(style="whitegrid")

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())

## Configuration

In [ ]:
SEED = 42
MAX_LENGTH = 256
USE_CLASS_WEIGHTS = True
CLASS_WEIGHT_POWER = 1.0

MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
MODEL_CANDIDATES = [
    MODEL_NAME,
    "dmis-lab/biobert-base-cased-v1.2",
]

ALLOWED_LABELS = [
    "rephrase",
    "delete",
    "ignore",
    "split",
]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sentence" / "raw"
TRAIN_PATH = DATA_DIR / "cochraneauto_sents_train.csv"
VAL_PATH = DATA_DIR / "cochraneauto_sents_val.csv"
TEST_PATH = DATA_DIR / "cochraneauto_sents_test.csv"

OUTPUT_DIR = PROJECT_ROOT / "models" / "biobert_action_classifier"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "biobert_action_classifier_predictions.csv"
ROUTED_OUTPUT_PATH = RESULTS_DIR / "biobert_action_classifier_routed_actions.csv"
PIPELINE_OUTPUT_PATH = RESULTS_DIR / "biobert_action_pipeline_outputs.csv"

BIOBART_LOCAL_DIR = PROJECT_ROOT / "models" / "biobart_sentence_no_context" / "best_model"
BIOBART_MODEL_CANDIDATES = [
    str(BIOBART_LOCAL_DIR),
    "GanjinZero/biobart-base",
    "facebook/bart-base",
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Preferred model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Use class weights: {USE_CLASS_WEIGHTS}")
print(f"Output directory: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")
print(f"Prediction path: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
print(f"Routed actions path: {ROUTED_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Pipeline output path: {PIPELINE_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

## Load Datasets

The classifier is trained on the provided Cochrane-auto `label` column. Only `complex` and `label` are used for classification. `merge` and `none` are removed because the planned sentence-level router supports only `rephrase`, `delete`, `ignore`, and `split`.

In [ ]:
def load_action_split(path: Path, split_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} file: {path}")
    raw_df = pd.read_csv(path)
    missing_columns = [column for column in ["complex", "label"] if column not in raw_df.columns]
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {missing_columns}")

    df = raw_df[["complex", "label"]].copy()
    df["complex"] = df["complex"].fillna("").astype(str).str.strip()
    df["label"] = df["label"].fillna("").astype(str).str.strip()
    df = df[df["complex"].ne("")].reset_index(drop=True)
    filtered_df = df[df["label"].isin(ALLOWED_LABELS)].reset_index(drop=True)
    return df, filtered_df

train_raw_df, train_df = load_action_split(TRAIN_PATH, "train")
val_raw_df, val_df = load_action_split(VAL_PATH, "validation")
test_raw_df, test_df = load_action_split(TEST_PATH, "test")

size_summary = pd.DataFrame([
    {"split": "train", "original_rows": len(train_raw_df), "filtered_rows": len(train_df)},
    {"split": "validation", "original_rows": len(val_raw_df), "filtered_rows": len(val_df)},
    {"split": "test", "original_rows": len(test_raw_df), "filtered_rows": len(test_df)},
])
display(size_summary)

for split_name, raw_df, filtered_df in [
    ("train", train_raw_df, train_df),
    ("validation", val_raw_df, val_df),
    ("test", test_raw_df, test_df),
]:
    print(f"Original {split_name} label distribution:")
    display(raw_df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))
    print(f"Filtered {split_name} label distribution:")
    display(filtered_df["label"].value_counts().rename_axis("label").reset_index(name="count"))

## Label Encoding

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(ALLOWED_LABELS)

label_to_id = {label: int(idx) for idx, label in enumerate(label_encoder.classes_)}
id_to_label = {int(idx): label for label, idx in label_to_id.items()}

for df in [train_df, val_df, test_df]:
    df["labels"] = label_encoder.transform(df["label"])

print("label_to_id:", label_to_id)
print("id_to_label:", id_to_label)

## Dataset Analysis

In [ ]:
def class_distribution(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = df["label"].value_counts().reindex(label_encoder.classes_, fill_value=0)
    result = counts.rename_axis("label").reset_index(name="count")
    result["percentage"] = (100 * result["count"] / len(df)).round(2)
    result["split"] = split_name
    return result

distribution_df = pd.concat([
    class_distribution(train_df, "train"),
    class_distribution(val_df, "validation"),
    class_distribution(test_df, "test"),
], ignore_index=True)
display(distribution_df)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=distribution_df, x="label", y="count", hue="split")
plt.title("Action label distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Class Weights

The training data is imbalanced, so the classifier uses weighted cross-entropy. This gives minority classes such as `ignore` and `split` more influence during training and should improve macro F1.

In [ ]:
train_class_counts = train_df["labels"].value_counts().reindex(range(len(label_encoder.classes_)), fill_value=0).sort_index()
raw_class_weights = len(train_df) / (len(label_encoder.classes_) * train_class_counts)
class_weights = raw_class_weights.pow(CLASS_WEIGHT_POWER)
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float)

class_weight_df = pd.DataFrame(
    {
        "label_id": train_class_counts.index,
        "label": [id_to_label[int(idx)] for idx in train_class_counts.index],
        "train_count": train_class_counts.values,
        "class_weight": class_weights.values,
    }
)
display(class_weight_df)
print("Class weights enabled:", USE_CLASS_WEIGHTS)

## Tokenization

Only the `complex` sentence is tokenized.

In [ ]:
def load_tokenizer(model_candidates: list[str]) -> tuple[Any, str]:
    errors = []
    for candidate in model_candidates:
        try:
            tokenizer = AutoTokenizer.from_pretrained(candidate)
            print(f"Loaded tokenizer: {candidate}")
            return tokenizer, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load tokenizer {candidate}: {exc}")
    raise RuntimeError("Could not load any tokenizer candidate.\n" + "\n".join(errors))

tokenizer, RESOLVED_MODEL_NAME = load_tokenizer(MODEL_CANDIDATES)

def tokenize_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    return tokenizer(
        examples["complex"],
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
    )

print("Resolved tokenizer:", RESOLVED_MODEL_NAME)
print("Example input:", train_df.loc[0, "complex"])
print("Example label:", train_df.loc[0, "label"], "->", train_df.loc[0, "labels"])

## Dataset Creation

In [ ]:
def make_dataset(df: pd.DataFrame) -> Dataset:
    dataset_df = df[["complex", "labels"]].copy()
    dataset = Dataset.from_pandas(dataset_df, preserve_index=False)
    dataset = dataset.map(tokenize_examples, batched=True)
    dataset = dataset.remove_columns(["complex"])
    dataset.set_format(type="torch")
    return dataset

train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

## Model

In [ ]:
def load_classifier_model(model_candidates: list[str], resolved_tokenizer_model: str):
    ordered_candidates = [resolved_tokenizer_model] + [
        candidate for candidate in model_candidates if candidate != resolved_tokenizer_model
    ]
    errors = []
    for candidate in ordered_candidates:
        try:
            model = AutoModelForSequenceClassification.from_pretrained(
                candidate,
                num_labels=len(label_encoder.classes_),
                id2label=id_to_label,
                label2id=label_to_id,
            )
            print(f"Loaded model: {candidate}")
            return model, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load model {candidate}: {exc}")
    raise RuntimeError("Could not load any classifier model candidate.\n" + "\n".join(errors))

model, RESOLVED_MODEL_NAME = load_classifier_model(MODEL_CANDIDATES, RESOLVED_MODEL_NAME)
model.to(device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Resolved model: {RESOLVED_MODEL_NAME}")
print(f"Model loaded on: {next(model.parameters()).device}")

## Metrics

In [ ]:
def compute_metrics(eval_pred) -> dict[str, float]:
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision_score(labels, predictions, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, predictions, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, predictions, average="weighted", zero_division=0),
    }

## Training

The classifier is fine-tuned with weighted cross-entropy to improve minority-class learning. The best checkpoint is selected by validation macro F1, which is more appropriate than accuracy for this imbalanced action classification task.

The best-model metric is configured as `f1_macro`; Hugging Face adds the `eval_` prefix internally during evaluation.

In [ ]:
def build_training_args() -> TrainingArguments:
    base_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        max_grad_norm=1.0,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        label_names=["labels"],
    )
    try:
        return TrainingArguments(evaluation_strategy="epoch", **base_kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="epoch", **base_kwargs)

training_args = build_training_args()


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def build_trainer() -> Trainer:
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    trainer_class = WeightedLossTrainer if USE_CLASS_WEIGHTS else Trainer
    extra_kwargs = {"class_weights": class_weights_tensor} if USE_CLASS_WEIGHTS else {}
    try:
        return trainer_class(processing_class=tokenizer, **extra_kwargs, **trainer_kwargs)
    except TypeError:
        return trainer_class(tokenizer=tokenizer, **extra_kwargs, **trainer_kwargs)

trainer = build_trainer()
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved best model to: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

## Training

The classifier is fine-tuned with weighted cross-entropy to improve minority-class learning. The best checkpoint is selected by validation macro F1, which is more appropriate than accuracy for this imbalanced action classification task.

In [ ]:
training_log_df = pd.DataFrame(trainer.state.log_history)
display(training_log_df)

validation_log_df = training_log_df[training_log_df["eval_f1_macro"].notna()].copy()
if len(validation_log_df):
    display(validation_log_df[["epoch", "step", "eval_accuracy", "eval_precision_macro", "eval_recall_macro", "eval_f1_macro", "eval_f1_weighted"]])
else:
    print("No validation metric rows found.")

## Test Evaluation

In [ ]:
test_metrics = trainer.evaluate(test_dataset, metric_key_prefix="test")
test_metrics_df = pd.DataFrame(
    [{"metric": metric, "score": value} for metric, value in test_metrics.items() if isinstance(value, (int, float))]
)
display(test_metrics_df)

## Predictions

In [ ]:
prediction_output = trainer.predict(test_dataset)
test_logits = prediction_output.predictions
test_pred_ids = np.argmax(test_logits, axis=-1)
test_true_ids = prediction_output.label_ids

test_predictions_df = pd.DataFrame({
    "complex": test_df["complex"].tolist(),
    "true_label": label_encoder.inverse_transform(test_true_ids),
    "predicted_label": label_encoder.inverse_transform(test_pred_ids),
})
test_predictions_df.to_csv(PREDICTION_PATH, index=False)
print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
display(test_predictions_df.head())

## Action Pipeline

The classifier output now triggers the actual sentence-level action:

- `delete` -> output an empty string
- `ignore` -> keep the original sentence
- `rephrase` -> generate a simplification with BioBART
- `split` -> generate with BioBART for now; later this can be replaced by a dedicated split model

This creates a concrete pipeline output, not only a classification table.

In [ ]:
ACTION_TO_PIPELINE_STEP = {
    "rephrase": "generate_with_biobart",
    "delete": "remove_sentence",
    "ignore": "keep_original_sentence",
    "split": "generate_with_biobart_future_split_model",
}

BIOBART_INPUT_PREFIX = "Rewrite this biomedical sentence in simpler language: "


def route_action(predicted_label: str) -> str:
    return ACTION_TO_PIPELINE_STEP[predicted_label]


def load_biobart_generator(model_candidates: list[str]):
    errors = []
    for candidate in model_candidates:
        if candidate == str(BIOBART_LOCAL_DIR) and not BIOBART_LOCAL_DIR.exists():
            print(f"Skipping local BioBART model; not found: {BIOBART_LOCAL_DIR.relative_to(PROJECT_ROOT)}")
            continue
        try:
            generation_tokenizer = AutoTokenizer.from_pretrained(candidate)
            generation_model = AutoModelForSeq2SeqLM.from_pretrained(candidate)
            generation_model.to(device)
            generation_model.eval()
            print(f"Loaded generator: {candidate}")
            return generation_tokenizer, generation_model, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load generator {candidate}: {exc}")
    raise RuntimeError("Could not load any BioBART/BART generator.\n" + "\n".join(errors))


generation_tokenizer, generation_model, RESOLVED_GENERATOR_NAME = load_biobart_generator(BIOBART_MODEL_CANDIDATES)


def generate_biobart_outputs(sentences: list[str], batch_size: int = 8) -> list[str]:
    outputs = []
    for start in range(0, len(sentences), batch_size):
        batch_sentences = sentences[start : start + batch_size]
        input_texts = [BIOBART_INPUT_PREFIX + sentence for sentence in batch_sentences]
        encoded = generation_tokenizer(
            input_texts,
            max_length=256,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            output_ids = generation_model.generate(
                **encoded,
                max_new_tokens=128,
                num_beams=4,
                length_penalty=0.9,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )
        outputs.extend(generation_tokenizer.batch_decode(output_ids, skip_special_tokens=True))
    return [output.strip() for output in outputs]


def execute_action_pipeline(predictions_df: pd.DataFrame) -> pd.DataFrame:
    pipeline_df = predictions_df.copy()
    pipeline_df["pipeline_action"] = pipeline_df["predicted_label"].map(route_action)
    pipeline_df["final_output"] = ""

    delete_mask = pipeline_df["predicted_label"].eq("delete")
    ignore_mask = pipeline_df["predicted_label"].eq("ignore")
    generate_mask = pipeline_df["predicted_label"].isin(["rephrase", "split"])

    pipeline_df.loc[delete_mask, "final_output"] = ""
    pipeline_df.loc[ignore_mask, "final_output"] = pipeline_df.loc[ignore_mask, "complex"]

    sentences_to_generate = pipeline_df.loc[generate_mask, "complex"].tolist()
    generated_outputs = generate_biobart_outputs(sentences_to_generate, batch_size=8) if sentences_to_generate else []
    pipeline_df.loc[generate_mask, "final_output"] = generated_outputs
    return pipeline_df


pipeline_outputs_df = execute_action_pipeline(test_predictions_df)
pipeline_outputs_df.to_csv(PIPELINE_OUTPUT_PATH, index=False)

# Keep this routing-only file too, because it is useful for debugging action decisions.
routed_actions_df = pipeline_outputs_df[["complex", "true_label", "predicted_label", "pipeline_action"]].copy()
routed_actions_df.to_csv(ROUTED_OUTPUT_PATH, index=False)

print(f"Generator used: {RESOLVED_GENERATOR_NAME}")
print(f"Saved routed actions to: {ROUTED_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved pipeline outputs to: {PIPELINE_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
display(pipeline_outputs_df.head(20))

## Run Pipeline on New Sentences

Use this section to classify new sentences and execute the corresponding action. `delete` returns an empty string, `ignore` keeps the original sentence, and `rephrase/split` call BioBART.

In [ ]:
def predict_actions(sentences: list[str]) -> pd.DataFrame:
    model.eval()
    encoded = tokenizer(
        sentences,
        max_length=MAX_LENGTH,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1).cpu().numpy()

    pred_ids = probabilities.argmax(axis=1)
    rows = []
    for sentence, pred_id, probs in zip(sentences, pred_ids, probabilities, strict=True):
        predicted_label = id_to_label[int(pred_id)]
        row = {
            "complex": sentence,
            "predicted_label": predicted_label,
            "pipeline_action": route_action(predicted_label),
            "confidence": float(probs[pred_id]),
        }
        for label, label_id in label_to_id.items():
            row[f"prob_{label}"] = float(probs[label_id])
        rows.append(row)
    return pd.DataFrame(rows)


def run_pipeline_on_new_sentences(sentences: list[str]) -> pd.DataFrame:
    action_df = predict_actions(sentences)
    action_df["true_label"] = "unknown"
    return execute_action_pipeline(action_df)

new_sentences = [
    "Computer reminders achieved a median improvement in process adherence of 4.2% across all reported process outcomes.",
    "Twenty-eight studies were included in the review.",
    "No adverse events were reported.",
    "The intervention improved symptoms and reduced hospital admissions compared with usual care.",
]

display(run_pipeline_on_new_sentences(new_sentences))

## Confusion Matrix and Classification Report

In [ ]:
labels_order = list(label_encoder.classes_)
cm = confusion_matrix(test_predictions_df["true_label"], test_predictions_df["predicted_label"], labels=labels_order)

plt.figure(figsize=(7, 5.5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels_order,
    yticklabels=labels_order,
)
plt.title("BioBERT Action Classifier Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

report = classification_report(
    test_predictions_df["true_label"],
    test_predictions_df["predicted_label"],
    labels=labels_order,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
display(report_df)

print("Split class metrics:")
display(report_df.loc[["split"]])

## Error Analysis

In [ ]:
misclassified_df = test_predictions_df[test_predictions_df["true_label"] != test_predictions_df["predicted_label"]].copy()

print(f"Misclassified examples: {len(misclassified_df)} / {len(test_predictions_df)}")
if len(misclassified_df):
    display(misclassified_df[["complex", "true_label", "predicted_label"]].sample(n=min(20, len(misclassified_df)), random_state=SEED))
else:
    print("No misclassified examples found.")

In [ ]:
correct_df = test_predictions_df[test_predictions_df["true_label"] == test_predictions_df["predicted_label"]].copy()

for label in labels_order:
    class_correct = correct_df[correct_df["true_label"] == label]
    print(f"Correctly classified {label}: {len(class_correct)}")
    if len(class_correct):
        display(class_correct[["complex", "true_label", "predicted_label"]].sample(n=min(10, len(class_correct)), random_state=SEED))

## Experiment Comparison

E0, E1, and E2 are text generation experiments, so they are evaluated with simplification metrics such as SARI, BLEU, and BERTScore. E3 is different: BioBERT/PubMedBERT is an action classifier, so it is evaluated with classification metrics such as Accuracy and Macro F1.

This means E3 is not directly comparable to E0/E1/E2 until we connect the classifier to the full simplification pipeline and generate final simplified text.

In [ ]:
def test_metric(name: str) -> float:
    return float(test_metrics.get(name, np.nan))

comparison_df = pd.DataFrame(
    [
        {
            "Experiment": "E0",
            "Model": "Llama 3.1 8B zero-shot",
            "Task": "generation",
            "SARI": 28.25,
            "BLEU": 3.6,
            "BERTScore F1": 0.897,
            "Accuracy": np.nan,
            "F1 Macro": np.nan,
            "F1 Weighted": np.nan,
        },
        {
            "Experiment": "E1",
            "Model": "FLAN-T5-base fine-tuned",
            "Task": "generation",
            "SARI": 26.86,
            "BLEU": 36.57,
            "BERTScore F1": 0.935,
            "Accuracy": np.nan,
            "F1 Macro": np.nan,
            "F1 Weighted": np.nan,
        },
        {
            "Experiment": "E2",
            "Model": "BioBART fine-tuned",
            "Task": "generation",
            "SARI": 31.86,
            "BLEU": 30.91,
            "BERTScore F1": 0.932,
            "Accuracy": np.nan,
            "F1 Macro": np.nan,
            "F1 Weighted": np.nan,
        },
        {
            "Experiment": "E3",
            "Model": f"{RESOLVED_MODEL_NAME} action classifier",
            "Task": "classification",
            "SARI": np.nan,
            "BLEU": np.nan,
            "BERTScore F1": np.nan,
            "Accuracy": test_metric("test_accuracy"),
            "F1 Macro": test_metric("test_f1_macro"),
            "F1 Weighted": test_metric("test_f1_weighted"),
        },
    ]
)
display(comparison_df)

## Documentation Notes

### Why Classify Actions Before Simplification

The classifier decides whether a sentence should be rewritten, deleted, kept, or split before any generator is used. This prevents BioBART from rewriting sentences that should be removed or kept unchanged.

### Delete vs Ignore

`delete` means the sentence should be removed from the simplified output. `ignore` means the sentence can stay in the simplified output without rewriting.

### Why Split Is Retained

`split` is rare but important because long biomedical sentences may need to become multiple simpler sentences. For now, split sentences can be routed to BioBART; later they can use a dedicated split generator.

### Why Merge and None Are Excluded

`merge` requires multi-sentence context, which is outside this sentence-level no-context experiment. `none` is not one of the planned simplification router actions.

### Integration With BioBART

The future pipeline is:

- `rephrase -> BioBART simplification`
- `delete -> remove sentence`
- `ignore -> keep original sentence`
- `split -> BioBART simplification now, future dedicated split model`

This notebook implements the classifier and a first action pipeline. BioBART generation is used for `rephrase` and `split`, while `delete` and `ignore` are handled deterministically.